# Memory

<div style="display: flex; justify-content: flex-start; gap: 10px;">
  <img src="./assets/LC_Memory_after.png" alt="Image 2" style="width:300px; border:1px solid #ccc; border-radius:6px;">
</div>

Persisting messages, or 'agent state' between invocations of the agent.

First, let's connect to our SQLite database containing music store data.


In [1]:
import { SqlDatabase } from "@langchain/classic/sql_db";
import { DataSource } from "typeorm";

const datasource = new DataSource({
    type: "sqlite",
    database: "./Chinook.db", // Replace with the link to your database
});
const db = await SqlDatabase.fromDataSourceParams({
    appDataSource: datasource,
});

Define a context schema to hold customer information (first and last name) that will be available throughout the agent's execution.


In [2]:
import { z } from "zod";

const contextSchema = z.object({
    db: z.instanceof(SqlDatabase)
});
type Context = z.infer<typeof contextSchema>;

Create a tool to execute SQL queries. It supports named parameters (`:first`, `:last`) that get replaced with values from the runtime context.


In [3]:
import { tool, Runtime } from "langchain";

const executeSQL = tool(async ({ query }, runtime: Runtime<Context>) => {
    return await db.run(query);
}, {
    name: "execute_sql",
    description: "Execute a SQLite command and return results.",
    schema: z.object({
        query: z.string()
    })
});

Define the system prompt that instructs the agent how to interact with the database safely and use named parameters.


In [4]:
export const SYSTEM = `You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool \`execute_sql\` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.`

Create the agent with our tools and system prompt. Note: **no checkpointer yet**, so the agent won't remember previous conversations.


In [5]:
import * as setup from "./setup.ts";
import { createAgent } from "langchain";

const agent = createAgent({
    model: "anthropic:claude-sonnet-4-5-20250929",
    tools: [executeSQL],
    systemPrompt: SYSTEM,
    contextSchema,
})

Disabling LANGSMITH_TRACING in Deno kernel.


## Repeated Queries

Ask about Frank Harris's last invoice. The agent successfully retrieves the information.


In [6]:
const stream = await agent.stream({
    messages: "This is Frank Harris, What was the total on my last invoice?",
}, {
    streamMode: "values",
    context: { db }
})

for await (const step of stream) {
    displayMessage(step.messages.at(-1))
}


┌────────────────────────────────────────────────────────────┐


│ 👤 HUMAN MESSAGE                                           │


└────────────────────────────────────────────────────────────┘


This is Frank Harris, What was the total on my last invoice?



┌────────────────────────────────────────────────────────────┐


...

[{"InvoiceId":374,"InvoiceDate":"2013-07-04 00:00:00","Total":5.94}]



┌────────────────────────────────────────────────────────────┐


│ 🤖 AI MESSAGE                                              │


└────────────────────────────────────────────────────────────┘


Hello Frank! The total on your last invoice (Invoice #374 from July 4, 2013) was **$5.94**.


Now ask a follow-up question: "What were the titles?" Without memory, the agent has no idea what we're referring to and asks for clarification.


In [7]:
const stream = await agent.stream({
    messages: "What were the titles?",
}, {
    streamMode: "values",
    context: { db }
})

for await (const step of stream) {
    displayMessage(step.messages.at(-1))
}


┌────────────────────────────────────────────────────────────┐


│ 👤 HUMAN MESSAGE                                           │


└────────────────────────────────────────────────────────────┘


What were the titles?



┌────────────────────────────────────────────────────────────┐


...

[{"Title":"For Those About To Rock We Salute You"},{"Title":"Balls to the Wall"},{"Title":"Restless and Wild"},{"Title":"Let There Be Rock"},{"Title":"Big Ones"}]



┌────────────────────────────────────────────────────────────┐


│ 🤖 AI MESSAGE                                              │


└────────────────────────────────────────────────────────────┘


Here are the first 5 album titles from the database:

1. For Those About To Rock We Salute You
2. Balls to the Wall
3. Restless and Wild
4. Let There Be Rock
5. Big Ones

Would you like to see more titles, or are you looking for titles from a specific context (like titles by a particular artist, or all titles)?


## Add memory

Let's fix this by adding a checkpointer. The `MemorySaver` will store conversation history so the agent can remember context across turns.


In [8]:
import * as setup from "./setup.ts";
import { createAgent } from "langchain";
import { MemorySaver } from "@langchain/langgraph";

const checkpointer = new MemorySaver()
const agent = createAgent({
    model: "anthropic:claude-sonnet-4-5-20250929",
    tools: [executeSQL],
    systemPrompt: SYSTEM,
    contextSchema,
    checkpointer
})

Ask the same question again, but this time with a `thread_id` to track the conversation.


In [9]:
const stream = await agent.stream({
    messages: "This is Frank Harris, What was the total on my last invoice?",
}, {
    streamMode: "values",
    configurable: { thread_id: "1" },
    context: { db }
})

for await (const step of stream) {
    displayMessage(step.messages.at(-1))
}


┌────────────────────────────────────────────────────────────┐


│ 👤 HUMAN MESSAGE                                           │


└────────────────────────────────────────────────────────────┘


This is Frank Harris, What was the total on my last invoice?



┌────────────────────────────────────────────────────────────┐


...

[{"InvoiceId":374,"InvoiceDate":"2013-07-04 00:00:00","Total":5.94}]



┌────────────────────────────────────────────────────────────┐


│ 🤖 AI MESSAGE                                              │


└────────────────────────────────────────────────────────────┘


Hello Frank Harris! The total on your last invoice (Invoice #374 from July 4, 2013) was **$5.94**.


Now when we ask "What were the titles?", the agent remembers the previous question about the invoice and returns the track titles! 🎵


In [10]:
const stream = await agent.stream({
    messages: "What were the titles?",
}, {
    streamMode: "values",
    configurable: { thread_id: "1" },
    context: { db }
})

for await (const step of stream) {
    displayMessage(step.messages.at(-1))
}


┌────────────────────────────────────────────────────────────┐


│ 👤 HUMAN MESSAGE                                           │


└────────────────────────────────────────────────────────────┘


What were the titles?



┌────────────────────────────────────────────────────────────┐


...

[{"Name":"Holier Than Thou"},{"Name":"Through The Never"},{"Name":"My Friend Of Misery"},{"Name":"The Wait"},{"Name":"Blitzkrieg"},{"Name":"So What"}]



┌────────────────────────────────────────────────────────────┐


│ 🤖 AI MESSAGE                                              │


└────────────────────────────────────────────────────────────┘


The titles on your last invoice were:

1. Holier Than Thou
2. Through The Never
3. My Friend Of Misery
4. The Wait
5. Blitzkrieg
6. So What
